# RoboBarista – LeRobot + RoboCasa (Google Colab)

**Ziel:** Eine existierende, auf RoboCasa trainierte **SmolVLA-Policy** in einer RoboCasa-/MuJoCo-Küche ausführen. Später prüfen wir a) wechselnde Startpositionen, b) unerwartete Positionsänderungen während einer Bewegung und c) messbare Positions- und Winkelfehler.

**Wichtig:** Dieses Notebook richtet zuerst die **offizielle** Integration ein und führt Basistests aus. Es behauptet **noch nicht**, dass die Policy verschobene Gegenstände präzise greift oder eine ganze Kaffeeabfolge zuverlässig ausführen kann. Für eine echte mehrstufige Manipulationssequenz und kontrollierte Eingriffe während der Bewegung folgt nach dem erfolgreichen Basistest ein eigener Test-Harness. Wir programmieren dabei **keine Greif-Policy** selbst.

**Versionen:** LeRobot `v0.6.1` (Python ≥ 3.12), robosuite `v1.5.2`, dein RoboCasa-Branch `barista-dev` (RoboCasa 1.0.1). Reproduzierbare Auswahl, aber Zusammenspiel auf Colab **erst im Test zu verifizieren**.

**Umgebung:** In Colab unter *Laufzeit → Laufzeittyp ändern* NVIDIA T4 wählen. Zellen nacheinander ausführen. Die Installation erfolgt **nur** unter `/content/robobarista-py312`; deine vorherige Python-3.11-Umgebung bleibt erhalten. Colab-Dateien sind temporär.

Offizielle Anleitung: https://huggingface.co/docs/lerobot/v0.6.1/robocasa (alternativ https://huggingface.co/docs/lerobot/main/robocasa). Policy: https://huggingface.co/lerobot/smolvla_robocasa.

## 1. GPU, RAM und freien Speicher prüfen

Der Test stoppt, falls keine NVIDIA-GPU oder weniger als 20 GiB freier Speicher zur Verfügung stehen.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

assert shutil.which("nvidia-smi"), "In Colab zuerst T4-GPU aktivieren: Laufzeit → Laufzeittyp ändern."
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Freier Speicher:", round(shutil.disk_usage("/content").free / 1024**3, 1), "GiB")
assert shutil.disk_usage("/content").free >= 20 * 1024**3, "Mindestens 20 GiB frei empfohlen. Keine Installation gestartet."
print("Aktuelles Notebook-Python:", subprocess.check_output(["python", "--version"], text=True).strip())

## 2. Projekt und offizielle Bibliotheken holen

Dein RoboCasa-Fork wird von `barista-dev` geladen. LeRobot und robosuite werden auf Tags festgelegt, damit nicht später stillschweigend andere Versionen geladen werden. Bereits vorhandene Verzeichnisse werden nicht überschrieben.

In [ ]:
import subprocess
from pathlib import Path

repos = [
    ("https://github.com/Coffee-Tobi/robocasa.git", "/content/robocasa", "barista-dev"),
    ("https://github.com/huggingface/lerobot.git", "/content/lerobot", "v0.6.1"),
    ("https://github.com/ARISE-Initiative/robosuite.git", "/content/robosuite", "v1.5.2"),
]
for url, folder, ref in repos:
    if not (Path(folder) / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", ref, url, folder], check=True)
    else:
        print("Bereits vorhanden (unverändert):", folder)
    print(folder, subprocess.check_output(["git", "-C", folder, "rev-parse", "--short", "HEAD"], text=True).strip())

## 3. Saubere Python-3.12-Umgebung anlegen

Die bisherige Python-3.11-Umgebung unter `/content/robocasa-env` bleibt bestehen; LeRobot 0.6.1 benötigt Python 3.12+. Die Installation dieser Umgebung muss nach einem vollständigen Colab-Laufzeitreset wiederholt werden.

In [ ]:
from pathlib import Path
import subprocess

MICRO = "/content/bin/bin/micromamba"
if not Path(MICRO).is_file():
    subprocess.run("mkdir -p /content/bin && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/bin bin/micromamba", shell=True, check=True)
PY_ENV = "/content/robobarista-py312"
if not Path(PY_ENV, "bin/python").is_file():
    subprocess.run([MICRO, "create", "-y", "-p", PY_ENV, "-c", "conda-forge", "python=3.12", "pip"], check=True)
PY = f"{PY_ENV}/bin/python"
print(subprocess.check_output([PY, "--version"], text=True).strip())

## 4. Offizielle Pakete installieren

**Es werden mehrere GB Pakete heruntergeladen.** Dieser Schritt kann in der kostenlosen Colab-Laufzeit einige Zeit dauern. Wir nutzen kein MimicGen und verändern Colabs Python 3.13 nicht.

Die RoboCasa-Installation erfolgt mit `--no-deps`, weil dessen `setup.py` LeRobot 0.3.3 erzwingt, obwohl wir LeRobot 0.6.1 brauchen. Die benötigten Runtime-Pakete installieren wir gezielt nach der offiziellen LeRobot-RoboCasa-Anleitung. Fehler bitte vollständig teilen, statt auf Verdacht weitere Pakete zu installieren.

In [ ]:
import os, subprocess

env = os.environ.copy()
env["PIP_NO_CACHE_DIR"] = "1"
env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

def pip_install(*args):
    print("\n>>> pip install", " ".join(args), flush=True)
    subprocess.run([PY, "-m", "pip", "install", "--no-cache-dir", *args], env=env, check=True)

pip_install("-e", "/content/lerobot[smolvla]")
pip_install("-e", "/content/robosuite")
pip_install(
    "numpy==2.2.5", "numba==0.61.2", "scipy==1.15.3", "mujoco==3.3.1",
    "pygame", "Pillow", "opencv-python", "pyyaml", "pynput", "tqdm",
    "termcolor", "imageio", "h5py", "lxml", "hidapi", "tianshou==0.4.10",
    "gymnasium",
)
pip_install("--no-deps", "-e", "/content/robocasa")
print("\nInstallation abgeschlossen. Als nächstes werden Imports und die GPU geprüft.")

## 5. GPU und Python-Imports in genau dieser Umgebung testen

Nur dann mit den Asset-Downloads weitermachen, wenn der Test erfolgreich ist.

In [ ]:
import os, subprocess

env = os.environ.copy()
env["MUJOCO_GL"] = "egl"
env["PYOPENGL_PLATFORM"] = "egl"
env["OPENBLAS_NUM_THREADS"] = "1"
check = """import sys, torch, mujoco, robosuite, robocasa, lerobot
print('Python:', sys.version.split()[0]); print('LeRobot:', getattr(lerobot,'__version__','importiert'))
print('RoboCasa:', getattr(robocasa,'__version__','importiert'))
print('robosuite:', robosuite.__version__); print('MuJoCo:', mujoco.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Torch erkennt T4 nicht'
print('GPU:',torch.cuda.get_device_name(0))
"""
subprocess.run([PY, "-c", check], env=env, check=True)

## 6. RoboCasa-Assets herunterladen (nur leichter Satz)

Offiziell benötigt die RoboCasa-GPU-Evaluation Texturen, generative Texturen, Lightwheel-Einrichtungsgegenstände und Lightwheel-Objekte. **Nicht** die vollständigen Objaverse-/Aigen-Pakete (~30 GB). Die Asset-Dateien werden auf Colab lokal gespeichert und nach einem Laufzeitreset gelöscht.

In [ ]:
import os, subprocess

env = os.environ.copy(); env["MUJOCO_GL"] = "egl"; env["PYOPENGL_PLATFORM"]="egl"
subprocess.run([PY, "-m", "robocasa.scripts.setup_macros"], env=env, check=True)
assets = [PY, "-m", "robocasa.scripts.download_kitchen_assets", "--type", "tex", "tex_generative", "fixtures_lw", "objs_lw"]
# RoboCasa fragt einmal zur Bestätigung. 'yes y' entspricht der offiziellen LeRobot-Docker-Anleitung.
subprocess.run("yes y | " + " ".join(assets), shell=True, env=env, check=True)

## 7. Vortrainiertes SmolVLA-RoboCasa-Modell herunterladen

Der eigentliche Checkpoint ist etwa **907 MB** groß (ohne mögliche zusätzliche Hugging-Face-Backbone-Dateien). Es wird nichts trainiert. Downloads liegen standardmäßig im temporären Hugging-Face-Cache. Ein späterer Evaluation-Aufruf lädt bei Bedarf weitere notwendige Modell-Dateien.

In [ ]:
import subprocess, os

env=os.environ.copy(); env["HF_HOME"]="/content/robobarista_hf_cache"
subprocess.run([PY, "-c", "from huggingface_hub import snapshot_download; p=snapshot_download('lerobot/smolvla_robocasa'); print('Checkpoint:',p)"], env=env, check=True)

## 8. Offizieller Basistest: eine RoboCasa-Aufgabe

**Erst dies bestätigt**, ob die Policy auf unserer T4 tatsächlich in MuJoCo läuft. Wir verwenden die **unveränderten offiziellen** `lerobot-eval`-Parameter einschließlich Kamera-Umbenennung. Zunächst **eine** Episode, um nicht unnötig GPU-Kontingent zu verbrauchen. Dies ist keine Präzisions- oder Live-Störungsmessung.

In [ ]:
import os, subprocess
from pathlib import Path

Path("/content/robobarista_runs").mkdir(exist_ok=True)
eval_env=os.environ.copy()
eval_env.update({
    "MUJOCO_GL":"egl", "PYOPENGL_PLATFORM":"egl",
    "OPENBLAS_NUM_THREADS":"1", "NUMBA_NUM_THREADS":"1",
    "HF_HOME":"/content/robobarista_hf_cache",
})
RENAME_MAP='{"observation.images.robot0_agentview_left": "observation.images.camera1", "observation.images.robot0_eye_in_hand": "observation.images.camera2", "observation.images.robot0_agentview_right": "observation.images.camera3"}'

def evaluate(task, n_episodes=1):
    cmd=[
        f"{PY_ENV}/bin/lerobot-eval",
        "--policy.path=lerobot/smolvla_robocasa",
        "--env.type=robocasa", f"--env.task={task}",
        "--eval.batch_size=1", f"--eval.n_episodes={n_episodes}",
        "--eval.use_async_envs=false", "--policy.device=cuda",
        f"--rename_map={RENAME_MAP}",
    ]
    print("Starte offiziellen Test:",task, flush=True)
    log_path=f"/content/robobarista_runs/{task}_eval.log"
    with open(log_path,"w") as log:
        proc=subprocess.Popen(cmd,env=eval_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,cwd="/content/lerobot")
        for line in proc.stdout:
            print(line,end=""); log.write(line);log.flush()
        ret=proc.wait()
    if ret:
        raise RuntimeError(f"Evaluation fehlgeschlagen (Code {ret}). Log: {log_path}")
    print("Basistest beendet. Log:",log_path)

evaluate("CloseFridge",n_episodes=1)

## 9. Als nächstes: Pick-and-Place mit variierenden Startpositionen

**Nur ausführen, wenn der Basistest oben funktioniert hat.** Die RoboCasa-Aufgabe `PickPlaceCounterToStove` ist im RoboCasa-Datensatzregister vorhanden. `n_episodes=3` führt unterschiedliche Episoden/Resets aus – die tatsächliche Variabilität der Gegenstandspositionen prüfen wir anschließend. Dies sind **separate** Episoden, noch keine verkettete Kaffee-Zubereitungssequenz.

Falls Lightwheel für eine Objektauswahl nicht ausreicht, kann die Umgebung scheitern. Dann bitte **keine 30 GB Assets blind installieren**, sondern zunächst das Fehlerprotokoll teilen.

In [ ]:
evaluate("PickPlaceCounterToStove",n_episodes=3)

## 10. Experimentplan nach erfolgreichem Basistest

Wir ergänzen **nur einen Versuchsaufbau**, keine neue Roboterbewegungslogik:

**A – Randomisierte Startpositionen:** mehrere Seeds, zufällig verschobene (physikalisch gültige) Tassen, Kännchen und Siebträger; Erfolgsquote pro Objekt erfassen.

**B – Störung im laufenden Ablauf:** Objekt in einem definierten Zeitschritt versetzen (oder durch einen physikalischen Stoß bewegen); die Policy erhält anschließend neue Kamerabilder. Wir prüfen, ob sie vor dem Greifen tatsächlich korrigiert. **Das ist mit diesem Notebook noch nicht implementiert.**

**C – Präzision:** Endposition und Orientierung gegen Sollziel messen, Kollisions- und Greiffehler gesondert protokollieren. Wir legen Toleranzen pro Aufgabe fest, statt pauschal „sehr präzise“ zu behaupten.

**D – Mehrere Objekte in Folge:** zunächst getrennte offizielle RoboCasa-Aufgaben mit SmolVLA testen; danach über eine Aufgabensteuerung mit Zustandsprüfung in **einer** Szene verketten. Die bestehende Policy ersetzt keine Zustandsmaschine und keine kraftsensitive Verriegelungsregelung.

**Bei Problemen:** Genau die fehlschlagende Zelle und ihre Ausgabe teilen. Keine unkontrollierten Downgrades. Ergebnisse/Logs können aus `/content/robobarista_runs` heruntergeladen oder nach Google Drive kopiert werden.